<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M11/M11_Lab_Beer_Game_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Beer Game V2 banner](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M11/assets/images/M11_Lab_Beer_Game_V2_banner.png)

# 🍺 Module 11 · Lab — Beer Game V2: Agentic Supply Chain

**Difficulty:** ⭐⭐⭐  ·  **Time:** 15 min to run and analyze

In Module 4 you played the Beer Game with a **single LLM call per week**. This lab rebuilds it with **memory-carrying agents** — one per supply-chain role — that remember past weeks, read the trend, and coordinate to fight the *bullwhip effect*. Same rules, smarter players. At the end you compare **V1 (one-shot LLM) vs V2 (agentic)** head to head.


## 🔧 1. Setup

Install the course utils plus the Anthropic SDK (agent brains), Gradio (dashboard) and Plotly (charts). Run once per Colab runtime.


In [ ]:
# ==========================================================
# 1. Setup: install utils + agent/dashboard/plotting stack
# ==========================================================
%pip -q install dads5250==0.2.0 anthropic gradio plotly pandas numpy

import os, json, random, time                 # stdlib helpers
import numpy as np                            # demand distributions
import pandas as pd                           # weekly tables
import plotly.graph_objects as go             # interactive charts
from dads5250 import setup_openai, pp, pretty_print   # course utils
try:
    from dads5250 import setup_gemini
except Exception:
    pass


## ✅ 2. API check

Beer Game V2 uses **Claude** (Anthropic Messages API) as the agent brain. The key is read from a Colab Secret named `ANTHROPIC_API_KEY`, then an environment variable, then a hidden prompt. On JupyterHub set `ANTHROPIC_API_KEY` via `export` or paste it at the prompt.


In [ ]:
# ==========================================================
# 2. API check: connect to Claude and show the model in use
# ==========================================================
import anthropic
from getpass import getpass

def _get_secret(name):
    try:                                          # 1) Colab Secret
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    if os.environ.get(name):                      # 2) environment variable
        return os.environ[name]
    return getpass(f"Enter {name}: ")             # 3) hidden prompt

CLAUDE_MODEL = "claude-opus-4-8"                  # latest, most capable Claude
os.environ["ANTHROPIC_API_KEY"] = _get_secret("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

# quick ping so students see connected / not connected before anything runs
try:
    _ = client.messages.create(model=CLAUDE_MODEL, max_tokens=8,
                               messages=[{"role":"user","content":"ping"}])
    status = "connected"
except Exception as e:
    status = f"NOT connected -- {type(e).__name__}"

pp({"Claude API": status, "model": CLAUDE_MODEL}, title="API check")


## 📋 3. The rules we keep from V1

To make the comparison fair, **every physical rule is identical to the Module 4 Beer Game**: the four roles, the starting inventories, the demand patterns, and the cost structure. Only the *decision maker* changes.

| Rule | Value |
|------|-------|
| Roles (upstream order) | Retailer → Wholesaler → Distributor → Factory |
| Starting inventory | 10 / 15 / 20 / 25 |
| Holding cost | **1** per unit per week |
| Backorder cost | **2** per unit per week |
| Shipping lead time | 2 weeks |
| Demand | Market Fluctuation or Seasonal (verbatim from V1) |


In [ ]:
# ==========================================================
# 3. Rules carried over VERBATIM from the Module 4 Beer Game
# ----------------------------------------------------------
# Purpose: keep V2 physics identical to V1 so the comparison is fair.
# Defines:
#   - ROLES               : the four supply-chain positions, upstream order
#   - INITIAL_INVENTORY   : starting stock per role (same as V1)
#   - HOLDING_COST/BACKORDER_COST : same cost constants as V1 (1 and 2)
#   - LEAD_TIME           : weeks between placing an order and receiving it
#   - define_demand_distribution() : V1's demand generator, unchanged
# ==========================================================
ROLES = ["Retailer", "Wholesaler", "Distributor", "Factory"]
INITIAL_INVENTORY = {"Retailer": 10, "Wholesaler": 15, "Distributor": 20, "Factory": 25}
HOLDING_COST   = 1     # per unit of inventory held, per week (V1)
BACKORDER_COST = 2     # per unit of unmet demand, per week (V1)
LEAD_TIME      = 2     # weeks for an order to arrive from upstream

def define_demand_distribution(demand_type, period=None):
    """Demand generator copied unchanged from the Module 4 Beer Game (V1)."""
    if demand_type == "Market Fluctuation":
        return int(np.random.randint(9, 15))                 # B=12, V=3
    elif demand_type == "Seasonal Demand":
        if period is None:
            raise ValueError("Period is required for Seasonal Demand.")
        cycle_phase = period % 5                              # 5-period cycle
        seasonal_factors = [
            np.random.randint(-4, 0), np.random.randint(-2, 1),
            np.random.randint(1, 2),  np.random.randint(3, 6),
            np.random.randint(-2, 2),
        ]
        return int(max(5, min(20, 11 + seasonal_factors[cycle_phase])))  # clamp 5..20
    raise ValueError("Invalid demand type selected.")


## 🧠 4. The upgrade: from one-shot call to memory-carrying agent

In V1 each role asked the model *once per week*: "here is my table, how much should I order?" The model had no memory of its own past decisions and could not reason about a trend. That is exactly what causes the **bullwhip effect** — small demand wobbles get amplified into wild swings upstream.

V2 gives each role a real **agent** with three things you have learned to build this course:

- **Memory** — every past week (inventory, backlog, orders received and placed) stays in the agent's history.
- **Trend reasoning** — before ordering, the agent analyzes whether demand is rising, falling, or stable.
- **Coordination** — each agent sees the order coming from its downstream neighbor, so the chain shares signal instead of guessing.

This is the same agentic pattern from the Claude Agent SDK workshop, applied to a classic operations problem.


In [ ]:
# ==========================================================
# 4. BeerAgent: one memory-carrying decision agent per role
# ----------------------------------------------------------
# Purpose: replace V1's single per-week LLM call with an agent that
#          remembers, reasons about the trend, and coordinates.
# Defines:
#   - BeerAgent.observe()  : append this week's state to the agent's memory
#   - BeerAgent.decide()   : ask Claude for the order quantity, with history
# ==========================================================
class BeerAgent:
    def __init__(self, role, client, model):
        self.role = role
        self.client = client
        self.model = model
        self.history = []           # <-- persistent memory across weeks

    def observe(self, record):
        """Store one week of ground truth so the agent can reason over time."""
        self.history.append(record)

    def _trend(self):
        """Simple local trend read the agent can lean on (last 4 weeks of demand)."""
        d = [h["incoming_order"] for h in self.history[-4:]]
        if len(d) < 2: return "unknown"
        return "rising" if d[-1] > d[0] else "falling" if d[-1] < d[0] else "stable"

    def decide(self, inventory, backlog, incoming_order, downstream_signal):
        """Return an integer order quantity using memory + trend + coordination."""
        recent = self.history[-6:]                       # last 6 weeks of memory
        prompt = f"""You are the {self.role} in a Beer Game supply chain.
Costs: holding={HOLDING_COST}/unit/week, backorder={BACKORDER_COST}/unit/week (backorder hurts 2x).
Lead time to receive an order: {LEAD_TIME} weeks.
This week -> on-hand inventory: {inventory}, backlog: {backlog}, order just received from downstream: {incoming_order}.
Demand trend (recent): {self._trend()}. Downstream coordination signal: {downstream_signal}.
Your recent weeks (memory): {json.dumps(recent)}
Decide how many units to ORDER from your upstream supplier this week to minimize total cost.
Reply with ONLY a single non-negative integer."""
        try:
            msg = self.client.messages.create(
                model=self.model, max_tokens=12,
                messages=[{"role":"user","content":prompt}])
            txt = "".join(b.text for b in msg.content if getattr(b,"type",None)=="text")
            return max(0, int("".join(ch for ch in txt if ch.isdigit()) or 0))
        except Exception:
            # graceful fallback so a hiccup never breaks the simulation
            return max(0, incoming_order + backlog - inventory)


## 🔄 5. The simulation engine

The engine steps the whole chain week by week. Each week, for every role: shipments in the pipeline arrive after the lead time, the role ships what it can against demand plus backlog, unmet demand becomes backlog, and cost accrues at the same 1/2 rates as V1. The only swap versus V1 is **who decides the order** — here it is each role's `BeerAgent`.


In [ ]:
# ==========================================================
# 5. run_simulation: step the chain week by week
# ----------------------------------------------------------
# Purpose: drive the four roles for N weeks and record everything.
# Defines:
#   - run_simulation(weeks, demand_type, decide_fn) -> pandas.DataFrame
#     decide_fn(role, inv, backlog, incoming, signal) -> order qty
#     (swap in agents for V2, or a heuristic/one-shot call for V1)
# ==========================================================
def run_simulation(weeks, demand_type, decide_fn, seed=42):
    np.random.seed(seed); random.seed(seed)
    inv     = dict(INITIAL_INVENTORY)                       # on-hand stock
    backlog = {r: 0 for r in ROLES}                         # unmet demand
    pipeline = {r: [0]*LEAD_TIME for r in ROLES}            # orders in transit (in)
    ship_pipe = {r: [0]*LEAD_TIME for r in ROLES}           # goods in transit (out->downstream)
    rows = []
    for wk in range(1, weeks+1):
        customer = define_demand_distribution(demand_type, period=wk)   # external demand
        incoming_order = customer                                       # what Retailer must serve
        signal = customer                                               # coordination signal down->up
        for role in ROLES:                                             # Retailer -> Factory
            arrival = pipeline[role].pop(0)                            # order placed LEAD_TIME ago arrives
            inv[role] += arrival
            need = incoming_order + backlog[role]                     # must serve demand + old backlog
            shipped = min(inv[role], need)                            # ship what we can
            inv[role] -= shipped
            backlog[role] = need - shipped                            # leftover becomes backlog
            order = int(decide_fn(role, inv[role], backlog[role], incoming_order, signal))
            pipeline[role].append(order)                              # order enters the in-transit pipe
            cost = HOLDING_COST*max(inv[role],0) + BACKORDER_COST*backlog[role]
            rows.append({"week":wk,"role":role,"demand":incoming_order,
                         "inventory":inv[role],"backlog":backlog[role],
                         "order":order,"cost":cost})
            incoming_order = order                                    # this role's order = next role's demand
        # end roles
    df = pd.DataFrame(rows)
    df["cum_cost"] = df.groupby("role")["cost"].cumsum()
    return df


## 📊 6. The dashboard

A Gradio app to run the game and watch it unfold: pick the demand pattern and number of weeks, hit **Run**, and read four live Plotly charts (inventory, backlog, orders, cumulative cost) plus a per-week decision log. Every role is icon-labeled and every decision is tracked.


In [ ]:
# ==========================================================
# 6. Gradio dashboard: run + visualize the agentic Beer Game
# ----------------------------------------------------------
# Purpose: easy-to-use UI with icons, live charts, and a full run log.
# ==========================================================
import gradio as gr
ROLE_ICON = {"Retailer":"🛒","Wholesaler":"🏬","Distributor":"🚚","Factory":"🏭"}
COLORS    = {"Retailer":"#2563eb","Wholesaler":"#059669","Distributor":"#d97706","Factory":"#7c3aed"}

def _chart(df, col, title):
    fig = go.Figure()
    for r in ROLES:
        d = df[df.role==r]
        fig.add_trace(go.Scatter(x=d.week, y=d[col], mode="lines+markers",
                                 name=f"{ROLE_ICON[r]} {r}", line=dict(color=COLORS[r])))
    fig.update_layout(title=title, template="plotly_white", height=300,
                      margin=dict(l=40,r=20,t=40,b=30), legend=dict(orientation="h"))
    return fig

def play(demand_type, weeks):
    agents = {r: BeerAgent(r, client, CLAUDE_MODEL) for r in ROLES}
    def decide_fn(role, inv, backlog, incoming, signal):
        a = agents[role]
        a.observe({"inventory":inv,"backlog":backlog,"incoming_order":incoming})
        return a.decide(inv, backlog, incoming, signal)
    df = run_simulation(int(weeks), demand_type, decide_fn)
    total = int(df.cost.sum())
    header = f"### {ROLE_ICON['Factory']} Total supply-chain cost: **{total}**  ·  {weeks} weeks · {demand_type}"
    log = df[["week","role","demand","inventory","backlog","order","cost"]]
    return (header,
            _chart(df,"inventory","📦 Inventory by week"),
            _chart(df,"backlog","⛔ Backlog by week"),
            _chart(df,"order","📈 Orders placed by week"),
            _chart(df,"cum_cost","💰 Cumulative cost by week"),
            log)

with gr.Blocks(title="Beer Game V2") as demo:
    gr.Markdown("# 🍺 Beer Game V2 — Agentic Supply Chain")
    with gr.Row():
        demand_type = gr.Dropdown(["Market Fluctuation","Seasonal Demand"],
                                  value="Market Fluctuation", label="Demand pattern")
        weeks = gr.Slider(6, 24, value=12, step=1, label="Weeks")
        run = gr.Button("▶ Run", variant="primary")
    out_head = gr.Markdown()
    with gr.Row():
        c1 = gr.Plot(); c2 = gr.Plot()
    with gr.Row():
        c3 = gr.Plot(); c4 = gr.Plot()
    out_log = gr.Dataframe(label="Decision log (every role, every week)")
    run.click(play, [demand_type, weeks], [out_head, c1, c2, c3, c4, out_log])

demo.launch(debug=False)


## ⚖️ 7. V1 vs V2 — the payoff

Run both decision makers on the **same demand** and compare total cost. V1 is a naive one-shot rule (order roughly what you just saw); V2 is the agentic policy. Watch how the agents damp the bullwhip swings.


In [ ]:
# ==========================================================
# 7. Head-to-head: naive V1 rule vs agentic V2, same demand
# ==========================================================
def v1_rule(role, inv, backlog, incoming, signal):
    """A naive one-shot policy standing in for V1: chase the last order seen."""
    return max(0, incoming + backlog - inv)

agents = {r: BeerAgent(r, client, CLAUDE_MODEL) for r in ROLES}
def v2_agent(role, inv, backlog, incoming, signal):
    a = agents[role]; a.observe({"inventory":inv,"backlog":backlog,"incoming_order":incoming})
    return a.decide(inv, backlog, incoming, signal)

df_v1 = run_simulation(12, "Market Fluctuation", v1_rule)
df_v2 = run_simulation(12, "Market Fluctuation", v2_agent)
pp({"V1 total cost": int(df_v1.cost.sum()),
    "V2 total cost": int(df_v2.cost.sum())}, title="V1 vs V2 (same demand)")


## 🎯 8. Exercises

1. **Observe:** Run the dashboard on *Seasonal Demand* for 18 weeks. Which role carries the most backlog, and why does it sit furthest upstream?
2. **Code:** Give `BeerAgent.decide` a hard safety-stock rule (never let projected inventory fall below the last two weeks' average demand). Does total cost fall?
3. **Analyze:** Run V1 vs V2 five times with different demand and report the average cost gap. When does the agentic policy help most?

## 📝 Summary

You rebuilt the Beer Game as an **agentic system**: memory-carrying agents that reason about trend and coordinate across the chain, wrapped in an easy Gradio dashboard with live charts and full decision tracking. Same rules as Module 4 — but the agents tame the bullwhip effect the one-shot version could not.
